![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 22 -- Lab 1: Convolutions and Pooling in PyTorch

Kaggle's garbage-classification photos (TrashNet): 2,527 pictures of cardboard, glass, metal, paper, plastic and trash, 512 x 384 pixels each. You use them as a box of real photos to see what today's layers do: the shape of a batch, the three channels, Sobel kernels through `F.conv2d`, a random `nn.Conv2d` and its feature maps, padding and stride against the formula, max pooling, parameter counts, the shift test, and then the deck's CNN written as a class and shape-traced before any training. Lab 2 trains one.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `asdasdasasdas/garbage-classification` (the TrashNet photos of Yang and Thung, 2016), about 43 MB, downloaded by `kagglehub` in the setup cell: one folder per class, loaded with `torchvision.datasets.ImageFolder` and resized to 96 x 128.

---
# Setup

In [ ]:
# kagglehub, torch, torchvision and matplotlib are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q kagglehub torch torchvision matplotlib

In [ ]:
import os
import kagglehub
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
torch.manual_seed(0)

In [ ]:
path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")
root = os.path.join(path, "Garbage classification", "Garbage classification")
print(os.listdir(root))
tf = transforms.Compose([transforms.Resize((96, 128)), transforms.ToTensor()])   # every photo -> (3, 96, 128), values 0..1
ds = datasets.ImageFolder(root, transform=tf)
CLASSES = ds.classes
loader = DataLoader(ds, batch_size=16, shuffle=True)
print(len(ds), CLASSES)

---
# Part 1 -- Images as tensors

The *Colour: Three Channels* and *Shapes in Code* slides.

## Task 1: A batch of photos

Take one batch with `xb, yb = next(iter(loader))` and print `xb.shape`, `yb.shape`, `xb.min().item()` and `xb.max().item()`: 16 photos, each `(3, 96, 128)`, values 0 to 1. Then show the first 8 photos in a row with their class names: `imshow` wants the channels last, so draw `xb[i].permute(1, 2, 0)`.

Syntax hint (the shape of the call, not the answer):

```python
image_hwc = image_chw.permute(1, 2, 0)   # (3, H, W) -> (H, W, 3) for imshow
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(xb.shape) == (16, 3, 96, 128) and tuple(yb.shape) == (16,), 'batch: DataLoader of the ImageFolder with batch_size=16; the transform gives (3, 96, 128)'
print('Task 1 passed')

## Task 2: The three channels

Take `img = xb[0]` (shape `(3, 96, 128)`). Show `img[0]`, `img[1]` and `img[2]` side by side with the colour maps `Reds`, `Greens` and `Blues` (`vmin=0, vmax=1`, axes off, titles). Then print `img[:, 50, 60]`: the three numbers of one pixel.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(img.shape) == (3, 96, 128) and tuple(img[:, 50, 60].shape) == (3,), 'channels: img[k] is one 96 x 128 grid; img[:, r, c] is one pixel across the three channels'
print('Task 2 passed')

---
# Part 2 -- Convolution

The *Hand-Made Kernels on a Real Photo*, *In Code* and *Counting the Parameters* slides.

## Task 3: Sobel kernels through F.conv2d

Make a grey version of the photo, `grey = img.mean(dim=0, keepdim=True)[None]` (shape `(1, 1, 96, 128)`: batch, channel, height, width). Build the vertical-edge kernel `K_v = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])[None, None]` (shape `(1, 1, 3, 3)`) and the horizontal one `K_h` (its transpose: rows `[-1, -2, -1]`, `[0, 0, 0]`, `[1, 2, 1]`). Compute `edges_v = F.conv2d(grey, K_v, padding=1)` and `edges_h` likewise, print their shapes, and show `grey[0, 0]`, `edges_v[0, 0].abs()` and `edges_h[0, 0].abs()` side by side in grey.

Syntax hint (the shape of the call, not the answer):

```python
out = F.conv2d(input_4d, kernel_4d, padding=1)   # input (B, C_in, H, W), kernel (C_out, C_in, k, k)
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(edges_v.shape) == (1, 1, 96, 128) and tuple(edges_h.shape) == (1, 1, 96, 128), 'F.conv2d: a (1, 1, 96, 128) input, a (1, 1, 3, 3) kernel, padding=1 keeps the size'
print('Task 3 passed')

## Task 4: A random Conv2d layer and its feature maps

Create `conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)` (8 kernels of 3 x 3 x 3, random at birth) and apply it to the whole batch: `fm = conv(xb)`. Print `fm.shape` (expected `(16, 8, 96, 128)`), `conv.weight.shape` (`(8, 3, 3, 3)`) and the number of parameters (`sum(p.numel() for p in conv.parameters())`, expected 224 = 8 * 27 + 8). Then show the 8 feature maps of the first photo, `fm[0, k].detach()`, in grey.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(fm.shape) == (16, 8, 96, 128) and n_conv == 224, 'Conv2d(3, 8, 3, padding=1): 8 output channels, 8 * (3 * 9) + 8 parameters'
print('Task 4 passed')

## Task 5: Padding, stride and the formula

For each `(k, p, s)` in `[(3, 0, 1), (3, 1, 1), (5, 2, 1), (3, 1, 2), (5, 0, 2)]`, build `nn.Conv2d(3, 8, kernel_size=k, padding=p, stride=s)`, apply it to `xb`, and print the output height and width next to the formula's answer, `(96 + 2 * p - k) // s + 1` and `(128 + 2 * p - k) // s + 1`. Keep `all_match = True` only if every row agrees.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert all_match, 'formula: floor((n + 2p - k) / s) + 1 for the height (96) and the width (128) separately'
print('Task 5 passed')

## Task 6: Max pooling

Create `pool = nn.MaxPool2d(2)` and apply it to the feature maps twice: `p1 = pool(fm)`, `p2 = pool(p1)`. Print the three shapes (`(16, 8, 96, 128)`, `(16, 8, 48, 64)`, `(16, 8, 24, 32)`) and show map 0 of the first photo at the three sizes (`.detach()`, grey). Does the pattern survive two poolings?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(p1.shape) == (16, 8, 48, 64) and tuple(p2.shape) == (16, 8, 24, 32), 'MaxPool2d(2) halves the height and the width, channels untouched'
print('Task 6 passed')

## Task 7: Count the parameters

Count the parameters of `nn.Conv2d(3, 16, 3)` and of `nn.Linear(3 * 96 * 128, 16)` as `n_c` and `n_l`, and print them with the formulas next to them: `16 * (3 * 9) + 16` and `3 * 96 * 128 * 16 + 16`. The convolution reads the same photo with 448 parameters; the flattened layer needs over half a million.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_c == 448 and n_l == 3 * 96 * 128 * 16 + 16, 'parameters: C_out * (C_in * k * k) + C_out for Conv2d; inputs * outputs + outputs for Linear'
print('Task 7 passed')

## Task 8: The shift test

Move the grey photo 20 pixels to the right with `torch.roll(grey, 20, dims=3)`, convolve it with `K_v` (padding 1) as `edges_shifted`, and check that it equals the original edge map rolled by 20: `torch.allclose(torch.roll(edges_v, 20, dims=3)[..., 40:-2], edges_shifted[..., 40:-2], atol=1e-4)` (the first 40 and last 2 columns are skipped: the roll wraps the border around and the padding sits elsewhere). Print the result. The same kernel finds the same pattern wherever it moved: shared weights.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert same, 'shift test: roll the input along dims=3 (width), convolve, compare with the rolled edge map on columns 40 to -2'
print('Task 8 passed')

---
# Part 3 -- The model as a class

The *In Code: The Model as a Class* and *The Pipeline, With Shapes* slides. Three blocks, because the photos are 96 x 128: the map goes 96 x 128, 48 x 64, 24 x 32, 12 x 16.

## Task 9: Write the model class

Write `class GarbageCNN(nn.Module)`. In `__init__(self, n_classes=6)`: call `super().__init__()`, then `self.features = nn.Sequential(...)` with three blocks, each `nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)` for channels 3 to 16, 16 to 32, 32 to 64; then `self.classifier = nn.Linear(64 * 12 * 16, n_classes)`. In `forward(self, x)`: `x = self.features(x)`, `x = x.flatten(1)`, `return self.classifier(x)`. Create `model = GarbageCNN()`, run `out = model(xb)`, and print `out.shape` (expected `(16, 6)`) and the parameter count `n_params`.

Syntax hint (the shape of the call, not the answer):

```python
class MyNet(nn.Module):
    def __init__(self, ...):
        super().__init__()
        self.features = nn.Sequential(...)   # the layers, declared once
        self.classifier = nn.Linear(...)
    def forward(self, x):
        ...                                  # how a batch flows through them
        return scores
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(out.shape) == (16, 6) and n_params == 448 + 4640 + 18496 + 64 * 12 * 16 * 6 + 6, 'GarbageCNN: three conv blocks 3->16->32->64 with padding 1 and MaxPool2d(2), then Linear(64 * 12 * 16, 6)'
print('Task 9 passed')

## Task 10: Trace the shapes

The shape habit: start from `h = xb[:1]` and, in a loop over `model.features`, apply each layer and print its class name with `tuple(h.shape)`. Then print `h.flatten(1).shape`: it must be `(1, 12288)`, the number the `Linear` layer was told.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert flat_shape == (1, 12288), 'trace: after three poolings the map is (64, 12, 16); flatten(1) gives 64 * 12 * 16 = 12288'
print('Task 10 passed')

## Task 11: The untrained network

Inside `torch.no_grad()`, compute `probs = torch.softmax(model(xb), dim=1)` and print `probs[0]` (six numbers close to 1/6) and the loss `nn.CrossEntropyLoss()(model(xb), yb).item()`: about `ln(6) = 1.79`, the number a loss curve starts from tomorrow in Lab 2.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert 1.5 < loss0 < 2.1 and abs(probs[0].sum().item() - 1) < 1e-4, 'untrained: softmax rows sum to 1, cross-entropy near ln(6)'
print('Task 11 passed')

---
## Task 12: Reflect

`F.conv2d` with your Sobel kernel and `nn.Conv2d` compute the same operation. What is the one difference between them, and what does training change? The shift test passed. Say in your own words why the feature map moved with the photo, and which of the deck's two words (local, shared) it illustrates. Your `Linear` layer was told 12,288. Where did that number come from, and what would the error message be if you had typed 64 * 24 * 32 instead?

*Your answers here*

---
## Conclusion

- **Images as tensors**: a batch of real photos is (16, 3, 96, 128), channels first, values 0 to 1; `permute` only for drawing.
- **Convolution in PyTorch**: `F.conv2d` ran your Sobel kernels, `nn.Conv2d` owns 8 random kernels and gave 8 feature maps; padding and stride matched the formula every time; a shifted photo gave a shifted map.
- **Pooling and parameters**: `MaxPool2d(2)` halved the maps twice and kept the pattern; a convolution needs 448 parameters where the flattened layer needs over half a million.
- **The model as a class**: `GarbageCNN` with `__init__` and `forward`, shape-traced to (64, 12, 16) and 12,288 inputs, giving 6 scores and an untrained loss of ln(6).

---

Made By **Sattam Altwaim**